# 노드 C — tonight

**6 GPU = 3노드 x 2 GPU. 이 노트북은 노드 C 전용.**

세 노드 전부 `exp5_tonight.py` 하나만 부른다. 잡 정의 · 우선순위 · 스킵 · 집계가 전부 거기 있다.
**세 노트북은 내용이 동일하고 노드 문자만 다르다** — 학습으로 배정되든 eval로 배정되든 이 하나로 된다.

**예산**: eval 1셀(LIBERO-10 x 50ep/task = 500ep) ≈ 2 GPU-h · 학습 1잡(150k) ≈ 8 GPU-h

---

## 판정 현황 (8/25 오후)

**① act+TE 열 측정 완료** (`exp3_act_te_ksweep`, 100ep/task, seed0, 동일 UBAI):

| K | act+TE | bimamba+TE | +carry |
|---|---|---|---|
| 10 | 72.5 | 75.4 | 74.3 |
| 15 | 67.3 | 75.6 | **76.4** |
| 20 | 61.2 | 71.4 | 71.9 |
| 50 | 39.7 | 64.1 | 63.6 |
| 100 | 27.7 | 49.3 | 50.2 |
| 150 | 21.9 | 26.8 | 27.1 |

**전 K에서 우리가 위. max 76.4(K=15) vs ACT max 72.5(K=10) → 100ep 기준 토너먼트 승리.**

**② stride sweep 슬라이드는 aloha 오염 확정** (은지님 8/24: "76까지 나온 건 libero에 없다").
ACT 76.0(K=100 s50)·BiMamba 75.0(s75) 폐기. **ACT 진짜 libero 최고 = 67.6(K=50 s10, TE off)**.
→ 레짐 맵(rm_*) 셀 = 그 실험의 UBAI 재구현 (은지님이 요청한 "해당 실험 다시 구현").

## 오늘 밤 할 일 — 확정 런

1. **te_* 500ep 재측정** — K=10/15 마진(+2.9/+9.1)은 bimamba 쪽 n=100(SE ±4.3)이라 오차 안.
   500ep(SE ±2.2)로 다시 재서 확정한다. exp3는 `ak{K}_te` 태그라 exp5가 `te_*`로 다시 도는
   게 맞다 (중복 아님, n 업그레이드).
2. **`bimamba_pure_k10/15` 학습** (critical, 노드 1대 선점) — 현재 bimamba 값은
   carry-학습 오염판(cpoff). act_k10/15는 이미 학습돼 있어서 critical에서 자동 이탈했다.
3. **rm K=50 s=10/25** — 67.6과 TE 없는 정면 승부 + carry 표 모순(69.5 vs 64.6) 해소.


## 1) 부팅

In [ ]:
import sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

NODE = 'C'
# 이 노드에서 쓸 GPU. 한 노드에서 창을 2개 띄울 때만 [2, 3] 처럼 직접 지정.
GPUS = v23.available_gpus([0, 1])
print('NODE', NODE, '| GPUS', GPUS)


## 2) 인벤토리 — 뭐가 학습돼 있고 뭐가 없나

서버 파일시스템을 실제로 스캔한다. `MISS` = 학습 필요, `PART` = 중단됨(resume 대상).
`MISS`/`PART`인 태그를 참조하는 eval 셀은 자동으로 큐에서 빠진다.

In [ ]:
rows = X.inventory()


## 3) 역할 배정 ← **여기가 오늘 뭘 할지 정한다**

규칙: ① critical 학습(`bimamba_pure_k10/15`)이 노드를 먼저 선점 →
② 남는 노드는 ready eval 큐 포화 → ③ 그래도 남으면 일반 학습.
예상 배정: **1노드 = 학습(pure), 2노드 = eval.** critical이 끝나면 전부 eval로 풀린다.

출력이 "기본값과 다르다"면 알려주는 `X.ROLE_OVERRIDE = ...` 한 줄을 **세 노트북 모두**에 넣어야 잡이 안 겹친다.

In [ ]:
roles = X.suggest()

# 위 출력이 "기본값과 다르다" 라고 하면, 알려주는 한 줄을 **세 노트북 모두**에 붙여넣고
# 이 셀부터 다시 실행할 것. (세 노드가 같은 역할표를 봐야 잡이 안 겹친다)
# X.ROLE_OVERRIDE = {'C': 'train'}


## 4) 계획 — 이 노드 몫

실행 전에 목록을 눈으로 확인할 것.

In [ ]:
plan = X.plan(NODE, GPUS)


## 5) preflight (약 12분) — **셋 중 한 노드에서 한 번만**

override(`n_action_steps`/`temporal_ensemble_coeff`)가 실제로 먹는지 확인.
exp3가 같은 방식(TE override)으로 이미 정상 완주했으므로 사실상 통과 확인용이다.
다른 노드에서 이미 통과했으면 건너뛸 것.

In [ ]:
X.preflight(gpu=GPUS[0], n_ep=5)


## 6) eval — 3)에서 `eval`로 배정됐을 때

먼저 dry-run으로 커맨드를 보고 실행. `GPUS` 수만큼 청크로 돌고 청크마다 블로킹한다.
로그는 `outputs/final/_logs/exp5__*.log`. **아침에 다시 실행하면 완료분은 skip되고 이어서 돈다.**

In [ ]:
X.run_evals(plan['eval'][:2], plan['gpus'], dry=True)


In [ ]:
if X.role_of(NODE) != 'eval':
    print('노드 ' + NODE + ' 는 train 으로 배정됐다 -> 7)번 학습 셀을 쓸 것. 여기는 건너뛴다.')
else:
    X.run_evals(plan['eval'], plan['gpus'])


## 7) 학습 — 3)에서 `train`으로 배정됐을 때

**dry-run에서 반드시 확인**: `bimamba_pure_*` 커맨드에 `--use_chunk_pairs`가 **없고** `--policy.sscp_enabled=false`가 **있어야** 한다. 이거 하나 틀리면 8시간을 날린다.

실행 셀은 잡이 끝날 때까지(~8h/잡) 블로킹한다.

In [ ]:
X.run_trains(plan['train'], plan['gpus'], dry=True)


In [ ]:
if X.role_of(NODE) != 'train':
    print('노드 ' + NODE + ' 는 eval 로 배정됐다 -> 6)번 eval 셀을 쓸 것. 여기는 건너뛴다.')
elif not plan['train']:
    print('학습 큐가 비었다.')
else:
    X.run_trains(plan['train'], plan['gpus'])


## 8) 이어서 — 다음 배치

위 배치가 끝나면 인벤토리가 바뀐다. 이 셀로 역할·계획을 다시 뽑고 6) 또는 7)로 돌아간다.

In [ ]:
# 위 배치가 끝나면 인벤토리가 바뀐다. 이 셀로 역할/계획을 다시 뽑고 6) 또는 7)로 돌아간다.
_ov = dict(X.ROLE_OVERRIDE)      # reload 하면 초기화되므로 수동 오버라이드를 보존한다
X = importlib.reload(X)
cf, v23 = X.setup(verbose=False)
X.ROLE_OVERRIDE = _ov
roles = X.suggest()
plan = X.plan(NODE, GPUS)


## 아침에 볼 것 — 500ep 확정 판정

`X.report()` 가 TE 표 + 레짐 맵을 전부 찍는다. 판단 기준:

| 결과 | 다음 |
|---|---|
| 500ep에서 K=10~20 마진 유지 | **토너먼트 승리 확정** → 이기는 셀만 seed/rep 추가로 굳히기 |
| K=10/15 마진 뒤집힘 | K=20/50 마진(+10.7/+24.4)으로 후퇴 — 크로스오버 스토리는 그대로 성립 |
| rm K=50 s10: `carry` > 67.6 | **TE 없는 절대 우위** — 동일 추론 비용 승리라 최상 |
| 순수 `bimamba`가 cpoff보다 크게 낮음 | carry-학습이 성능원이었다는 뜻 → bimos 중심으로 표 재구성 |

**주의**: 500ep 기준 SE ≈ ±2.2%p. 5%p 미만 차이는 seed/rep을 늘린 뒤에만 주장할 것.

In [ ]:
X.report()
